# APS Failure (Scania Trucks) - preprocessing
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadDevX/ML-Research-Dataset-Imbalance/blob/main/aps/00_preprocessing/aps_preprocessing.ipynb)

Stateless steps only: label mapping, removing identifiers and duplicates, marking invalid values
as missing, and (for large data) a fixed stratified sample. Everything that is *fitted*
(imputation, scaling, TF-IDF, resampling) happens later inside each training fold.

Writes `data/processed/aps.parquet` and the fixed 5x2 outer folds `data/splits/aps_outer.npz`.
Both are committed, so every experiment notebook (and your teammate) uses identical data and folds.
Raw data is expected in `data/raw/` (not committed; see `data/README.md`).

In [1]:
# ---- Setup: identical in every notebook. Do not edit. ----
import os, sys, subprocess
REPO_URL = "https://github.com/MuhammadDevX/ML-Research-Dataset-Imbalance.git"
BRANCH = "main"
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_ROOT = "/content/ML-Research-Dataset-Imbalance"
    if not os.path.isdir(REPO_ROOT):
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_ROOT], check=True)
    else:
        subprocess.run(["git", "-C", REPO_ROOT, "pull", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    f"{REPO_ROOT}/requirements.txt"], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    # Results go to Google Drive so they survive a disconnect.
    os.environ.setdefault("RESULTS_ROOT", "/content/drive/MyDrive/ML-Research-Dataset-Imbalance")
else:
    REPO_ROOT = os.path.abspath(os.getcwd())
    while not os.path.isdir(os.path.join(REPO_ROOT, "common")):
        parent = os.path.dirname(REPO_ROOT)
        if parent == REPO_ROOT:
            raise RuntimeError("Open this notebook from inside the repository.")
        REPO_ROOT = parent

sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
from common.env import check_environment
check_environment()


Environment OK: scikit-learn 1.9.1, imbalanced-learn 0.14.2, xgboost 3.4.1, lightgbm 4.7.0


## Load
APS Failure and Operational Data for Scania Trucks (UCI, Scania CV AB 2016, GPL-3.0).
Two files, `aps_failure_training_set.csv` (60,000 rows) and `aps_failure_test_set.csv`
(16,000 rows), each with a 20-line licence header, 170 anonymised numeric sensor
features and `class` = `pos` (failure of the Air Pressure System) or `neg` (failure
of another component). Missing values are written as `na`.

The two files are the dataset's own train/test split. This study uses its own 5x2
outer folds, so both files are pooled; the source file is kept as `meta_source`
(description only, never a feature).

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from common import data, settings
from common.paths import RAW_DIR

parts = []
for part in ("training", "test"):
    d = pd.read_csv(RAW_DIR / "aps" / f"aps_failure_{part}_set.csv", skiprows=20, na_values="na")
    parts.append(pd.concat([d, pd.Series(part, index=d.index, name="meta_source")], axis=1))
raw = pd.concat(parts, ignore_index=True)
print(raw.shape)
raw["class"].value_counts()

(76000, 172)


class
neg    74625
pos     1375
Name: count, dtype: int64

## Clean (stateless steps only)
* Label: `pos` -> 1 (minority, APS failure), `neg` -> 0.
* `na` -> NaN (imputed later inside each training fold). No columns are dropped here:
  constant columns are removed by `VarianceThreshold` inside the pipeline.
* Remove duplicate rows, then rows whose features appear with both labels.
* **Stratified 20,000-row sample** (seed `MASTER_SEED`), keeping the natural failure
  rate. The full 76,000 x 170 table makes Random Forest after SMOTE about 6 minutes per
  single fit; the sample keeps the whole factorial and E2 on one workstation (same
  approach as CIC-IDS2017 and Credit Card Fraud).

In [3]:
df = raw.copy()
df["y"] = (df.pop("class") == "pos").astype(int)
feats = [c for c in df.columns if c not in ("y", "meta_source")]
print("missing cells:", int(df[feats].isna().sum().sum()),
      f"({df[feats].isna().to_numpy().mean():.1%})")
print("duplicate rows:", int(df.duplicated(subset=feats + ["y"]).sum()))
df = df.drop_duplicates(subset=feats + ["y"])
conflict = df.duplicated(subset=feats, keep=False)
print("rows with both labels:", int(conflict.sum()))
df = df[~conflict].reset_index(drop=True)

APS_N = 20_000
keep, _ = train_test_split(np.arange(len(df)), train_size=APS_N, stratify=df["y"],
                           random_state=settings.MASTER_SEED)
df = df.iloc[np.sort(keep)].reset_index(drop=True)
print(df.shape)
df["y"].value_counts()

missing cells: 1078695 (8.3%)


duplicate rows: 0


rows with both labels: 0
(20000, 172)


y
0    19638
1      362
Name: count, dtype: int64

## Save processed data and the 5x2 outer folds

In [4]:
data.save_processed("aps", df)
data.make_outer_splits(df["y"], "aps")
data.summary("aps")

{'dataset': 'aps',
 'n': 20000,
 'd': 170,
 'n_minority': 362,
 'minority_pct': 1.81,
 'IR': 54.25,
 'nan_cells': 285342}